# PyBaMM Fast Charging, Cell Design and Parameter Estimation - Exercises

Ten senior-level problems built around three daily workflows:

1. **Fast-charge protocols**: time-to-80 %, lithium-plating limits, multi-stage charging.
2. **Cell design optimisation**: electrode thickness vs. rate, porosity/thickness
   Pareto fronts, plating-limited areal loading.
3. **Parameter estimation**: ECM pulse fitting, GITT diffusivity (analytical and
   model-based), degradation-mode analysis from a C/20 curve.

Every problem runs real PyBaMM simulations on published parameter sets
(`Chen2020` = LG M50 21700, NMC811 | graphite-SiOx, 5 Ah; `Xu2019` = NMC532 half-cell).
Each simulation takes well under a few seconds.

**How this notebook works**

* The **markdown** above each problem explains the physics, why it matters, the
  formula and the exact function contract (name, arguments, return value).
* The **code cell** under it is deliberately almost empty: you write the whole
  PyBaMM workflow yourself (model, `ParameterValues`, `Experiment`, `Simulation`,
  `solve`, post-processing).
* Stuck? Open the collapsed **Hints** one at a time. They get more specific as
  you go, so try to stop at the first one that unblocks you.
* Every **test cell** already shows its **target output**, captured by running the
  reference solution. Run it after implementing your function and compare.
  Small differences in the last digit are fine.
* Cells marked **Virtual lab** are given scaffolding: they generate synthetic
  "measured" data with a known truth so you can check your estimates. Run them as
  they are.
* The placeholders return dummy values, so the notebook runs top to bottom even
  before you have solved anything.

The companion **Solutions** notebook has fully worked versions of every problem.

In [4]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import brentq, curve_fit

import pybamm

pybamm.set_logging_level("WARNING")

## 1. Fast-charge protocols

Fast-charge capability is limited by three coupled things: **polarisation**
(the cell hits the upper cut-off voltage early and the CV phase takes over),
**lithium plating** at the anode/separator interface (the anode potential drops
below 0 V vs Li/Li$^+$), and **heat**. All three problems below use the DFN,
because the plating criterion needs the through-thickness electrolyte potential
that the SPM does not resolve.

#### Problem 1: Time to 80 % SOC under CC-CV, and why more C-rate stops helping

**Why it matters.** "10-80 % in X minutes" is the headline fast-charge KPI on every
customer specification. Under CC-CV, raising the CC current shortens the CC phase, but
polarisation makes the cell reach the upper cut-off voltage at a *lower* SOC, so a
growing share of the 10-80 % window is delivered in the slow, tapering CV phase. Beyond
some C-rate you are paying in heat and plating risk for almost no time saved.

**Physics.** By coulomb counting from a known start SOC:

$$\mathrm{SOC}(t) = \mathrm{SOC}_0 + \frac{1}{Q_{nom}}\int_0^t |I(\tau)|\,d\tau,
\qquad t_{80} = \min\{t : \mathrm{SOC}(t) = 0.8\}$$

$$\text{CV share} = \frac{\max(0,\ t_{80} - t_{CC,end})}{t_{80}} \times 100\ \%$$

**Your task.** Write `cccv_time_to_soc(c_rate, target_soc=0.8, start_soc=0.1)` returning
a dict with keys:

* `"time_to_target_min"`: time from the start of charge to `target_soc` [min]
* `"cv_share_pct"`: the CV share defined above [%]

Set-up: DFN, `Chen2020`, start at `start_soc`, charge at `c_rate` C to 4.2 V, then hold
4.2 V until C/20.

<details>
<summary><b>Hint 1</b>: building the simulation</summary>

Both charge steps belong to *one* cycle, so pass them to the `Experiment` as a single
tuple. The start SOC is given at solve time, not in the parameter values.

</details>

<details>
<summary><b>Hint 2</b>: getting SOC out of the solution</summary>

`"Discharge capacity [A.h]"` is cumulative and **negative while charging**. Combine it
with `"Nominal cell capacity [A.h]"` from the parameter values and `"Time [s]"`. SOC
increases monotonically, so `np.interp` can invert it to find $t_{80}$.

</details>

<details>
<summary><b>Hint 3</b>: when did the CC phase end?</summary>

Each experiment step has its own sub-solution: `solution.cycles[0].steps[0]` is the CC
step; the last value of its `"Time [s]"` is $t_{CC,end}$.

</details>

In [ ]:
def cccv_time_to_soc(c_rate, target_soc=0.8, start_soc=0.1):
    """Return {"time_to_target_min", "cv_share_pct"} for a DFN CC-CV charge of Chen2020."""
    model = pybamm.lithium_ion.DFN()
    parameter_values = pybamm.ParameterValues("Chen2020")
    experiment = pybamm.Experiment(
    [
        (
            f"Charge at {c_rate}C until 4.2V",
            "Hold at 4.2V until C/20",
        )
    ]
)
    sim = pybamm.Simulation(model, parameter_values=parameter_values,experiment=experiment)
    sol = sim.solve(initial_soc=start_soc)
    soc = sol["Throughput capacity [A.h]"].data / parameter_values["Nominal cell capacity [A.h]"] + start_soc
    t = sol["Time [s]"].data / 60
    t_target = np.interp(target_soc, soc, t)
    t_cc_end = sol.cycles[0].steps[0]["Time [s]"].data[-1] / 60
    cv_share = (max(0,t_target - t_cc_end) / t_target)*100
    # return sol
    return soc, sol, {"time_to_target_min": t_target, "cv_share_pct": cv_share}

In [ ]:
model = pybamm.lithium_ion.DFN()
# model.variables.search("capacity")
names = list(model.variables.keys())
# list(model.default_parameter_values.keys())
# model.default_parameter_values.search("temperature")

In [ ]:
model.variables.search("potential")

In [ ]:
soc, sol, r = cccv_time_to_soc(1.0, target_soc=0.8, start_soc=0.1)
# help(sol.cycles[0])
# len(sol.cycles)
# sv = sol.summary_variables
# names = sv.all_variables

_Run the cell below, then compare your output to the target output already shown beneath it._

In [ ]:
crates = [0.5, 0.7, 1.0, 2.0, 3.0, 4.0]

fig,axes = plt.subplots(nrows=len(crates), ncols=2, figsize=(6, 2*len(crates)))

rows = []
for i, c in enumerate(crates):
    soc, sol, r = cccv_time_to_soc(c,target_soc=0.8, start_soc=0.1)
    rows.append({"C-rate": c,
                 "t(10-80%) [min]": round(r["time_to_target_min"], 1),
                 "CV share [%]": round(r["cv_share_pct"], 1)})
    
    current = sol["Current [A]"].data
    voltage = sol["Terminal voltage [V]"].data
    capacity = sol["Discharge capacity [A.h]"].data
    with np.errstate(divide="ignore", invalid="ignore"):
        dqdv = np.gradient(capacity, voltage)      
    
    # Plot
    plot = pybamm.QuickPlot(
        sol, ["Current [A]", "Voltage [V]"],
        time_unit="minutes",
    )    
    plot.plot(0)
    plot.axes.by_variable("Current [A]").axvline(r["time_to_target_min"], color="k", linestyle=":")
    plot.axes.by_variable("Voltage [V]").axvline(r["time_to_target_min"], color="k", linestyle=":")

    ax_soc, ax_dqdv = axes[i]
    ax_soc.plot(sol["Time [s]"].data / 60, soc)
    ax_soc.axvline(r["time_to_target_min"], color="k", linestyle=":")
    ax_soc.axhline(0.8, color="k", linestyle=":")
    ax_dqdv.plot(-capacity, dqdv)
    # ax_dqdv.set_xlim(voltage[0], 4.2)
    ax_dqdv.set_ylim(-10, 10)
    ax_dqdv.axvline(-0.8*capacity[-1], color="k", linestyle=":")



print(pd.DataFrame(rows).to_string(index=False))

#### Problem 2: Plating-limited charge C-rate

**Why it matters.** Lithium plating is the failure mode that sets the fast-charge
limit of high-energy cells. Plating becomes thermodynamically possible wherever the
anode's local solid-electrolyte potential difference drops below the Li/Li$^+$
potential. The worst location is the **anode/separator interface**: it carries the
highest local reaction current and sees the largest electrolyte potential drop. A model
gives you this internal signal, which you can never measure on a production cell, and
lets you turn it into a charge-rate limit.

**Physics.** The plating criterion (potentials vs. Li/Li$^+$):

$$\eta_{Li}(t) = \phi_s - \phi_e\Big|_{x = L_n}, \qquad
\text{plating margin} = \min_t\ \eta_{Li}(t)$$

Margin $> 0$: no plating. Margin $< 0$: plating is possible. The plating-limited C-rate
is the root of margin(C) = 0.

**Your task.** Write `min_plating_potential_mV(c_rate, ambient_c=25.0, start_soc=0.1)`
returning the plating margin in **mV** for a constant-current charge from `start_soc`
to 4.2 V (no CV: the minimum is reached at the end of CC). Use the **isothermal** DFN
with `Chen2020`, with the cell held at `ambient_c` °C. That is a conservative "perfect
cooling" assumption; Problem 3 adds self-heating.

The test cell root-finds the plating-limited C-rate at 25 °C and 10 °C with `brentq`.
For reference, the LG M50 datasheet limits charging to about 0.7C.

<details>
<summary><b>Hint 1</b>: temperature</summary>

An isothermal model runs at the ambient temperature, but set **both**
`"Ambient temperature [K]"` and `"Initial temperature [K]"` so they are consistent.
Convert from °C.

</details>

<details>
<summary><b>Hint 2</b>: the output variable</summary>

PyBaMM already provides $\phi_s - \phi_e$ evaluated at the separator side of the
negative electrode: search `model.variables` for
`"surface potential difference at separator interface"`. Take the minimum over time
and convert V to mV.

</details>

<details>
<summary><b>Hint 3</b>: caveat</summary>

`Chen2020` has only partial temperature dependence, so the 10 °C result
*underestimates* the real low-temperature penalty. Use a parameter set with measured
Arrhenius data before using this for a real low-T spec.

</details>

In [ ]:
c_rate = 0.2
start_soc=0.1

model = pybamm.lithium_ion.DFN()
parameter_values = pybamm.ParameterValues("Chen2020")
    # parameter_values.update({"Ambient temperature [K]": ambient_c + 273.15})
experiment = pybamm.Experiment(
[
    f"Charge at {c_rate}C until 4.2V",
])
sim = pybamm.Simulation(model, parameter_values=parameter_values,experiment=experiment)
sol = sim.solve(initial_soc=start_soc)

In [ ]:
phi_s = sol["Negative electrode potential [V]"].entries
phi_e = sol["Negative electrolyte potential [V]"].entries
current = sol["Current [A]"].data
voltage = sol["Terminal voltage [V]"].data

In [ ]:
L_n = parameter_values["Negative electrode thickness [m]"]
phi = sol["Negative electrode potential [V]"]
phi_t = phi(t=sol["Time [s]"].entries,x=L_n)

In [ ]:
sim.mesh["separator"].npts       # same nodes, from the mesh

In [ ]:
plt.plot(phi_t)


In [ ]:
model_params = model.get_parameter_info()

In [ ]:
model.variables.search("surface potential difference")

In [19]:
def min_plating_potential_mV(c_rate, ambient_c=25.0, start_soc=0.1):
    """Return the minimum anode/separator phi_s - phi_e [mV] during a CC charge to 4.2 V."""
    model = pybamm.lithium_ion.DFN()
    parameter_values = pybamm.ParameterValues("Chen2020")
    parameter_values.update({"Ambient temperature [K]": ambient_c + 273.15})
    experiment = pybamm.Experiment(
    [
        f"Charge at {c_rate}C until 4.2V",
    ])
    sim = pybamm.Simulation(model, parameter_values=parameter_values,experiment=experiment)
    sol = sim.solve(initial_soc=start_soc)
    phi_s = sol["Negative electrode potential [V]"].entries
    phi_e = sol["Negative electrolyte potential [V]"].entries
    eta = sol['Negative electrode surface potential difference at separator interface [V]'].entries
    # eta_2 = phi_s - phi_e
    return eta.min()*1000

In [ ]:
eta = min_plating_potential_mV(0.7, ambient_c=25.0, start_soc=0.1)

_Run the cell below, then compare your output to the target output already shown beneath it._

In [ ]:
table = pd.DataFrame(
    {f"{T:.0f} degC": [round(min_plating_potential_mV(c, T), 1) for c in [0.3, 0.7, 1.0]]
     for T in [25.0, 10.0]},
    index=["0.3C", "0.7C", "1.0C"],
)
print("Plating margin [mV]:")
print(table)
print()
for T in [25.0, 10.0]:
    try:
        c_lim = brentq(lambda c: min_plating_potential_mV(c, T), 0.1, 3.0, xtol=1e-3)
        print(f"Plating-limited charge rate at {T:4.1f} degC: {c_lim:.3f} C")
    except ValueError as e:
        print(f"Could not bracket a root at {T} degC (function not implemented yet?): {e}")

Plating margin [mV]:
      25 degC  10 degC
0.3C     37.9     13.7
0.7C     -1.1    -26.6
1.0C    -16.9    -42.2

Could not bracket a root at 25.0 degC (function not implemented yet?): f(a) and f(b) must have different signs
Could not bracket a root at 10.0 degC (function not implemented yet?): f(a) and f(b) must have different signs


#### Problem 3: Designing a multi-stage (ladder) fast-charge protocol

**Why it matters.** Problem 2 showed that the plating margin is worst near the end of
charge, where the anode is nearly full. A **multi-stage constant-current** (MSCC,
"ladder") protocol exploits that: high current while the anode is empty, stepping down
as the voltage (a proxy for SOC) rises. Done right, it can be *faster* than the
datasheet CC-CV **and** keep a *larger* plating margin. Self-heating also helps,
because a warmer cell has lower overpotentials.

**Your task.** Write `evaluate_charge_protocol(cc_stages, start_soc=0.1, target_soc=0.8)`.

* `cc_stages` is a list of `(c_rate, cutoff_voltage)` tuples, e.g. `[(1.5, 3.9), (1.0, 4.05), (0.5, 4.2)]`.
  Each stage charges at `c_rate` until `cutoff_voltage`. After the last stage, hold 4.2 V
  until C/20.
* Model: DFN with a **lumped thermal** model, `Chen2020`, starting at `start_soc`.
* Return a dict with:
  * `"time_to_target_min"`: time to `target_soc` [min] (as in Problem 1)
  * `"min_plating_mV"`: plating margin over the whole charge [mV] (as in Problem 2)
  * `"temp_rise_K"`: peak minus initial cell temperature [K]
  * `"soc"`, `"plating_mV"`: NumPy arrays (SOC and $\phi_s-\phi_e$ in mV over time) for plotting

The test cell compares the datasheet-style 0.7C CC-CV, a 1C CC-CV and a 3-step ladder.

<details>
<summary><b>Hint 1</b>: building the step list</summary>

Generate the CC step strings from `cc_stages` with a list comprehension, append the CV
hold, and wrap the whole list in a tuple so it is one cycle.

</details>

<details>
<summary><b>Hint 2</b>: thermal model and temperature output</summary>

Model options are passed as a dict when you create the model (key `"thermal"`). The
temperature you want is the x-averaged cell temperature in K.

</details>

In [6]:
model = pybamm.lithium_ion.DFN()
model.variables.search("temperature")

Results for 'temperature': ['Ambient temperature [K]', 'Volume-averaged ambient temperature [K]', 'Cell temperature [K]', 'Negative current collector temperature [K]', 'Positive current collector temperature [K]', 'X-averaged cell temperature [K]', 'Volume-averaged cell temperature [K]', 'Negative electrode temperature [K]', 'X-averaged negative electrode temperature [K]', 'Separator temperature [K]', 'X-averaged separator temperature [K]', 'Positive electrode temperature [K]', 'X-averaged positive electrode temperature [K]', 'Ambient temperature [C]', 'Volume-averaged ambient temperature [C]', 'Cell temperature [C]', 'Negative current collector temperature [C]', 'Positive current collector temperature [C]', 'X-averaged cell temperature [C]', 'Volume-averaged cell temperature [C]', 'Negative electrode temperature [C]', 'X-averaged negative electrode temperature [C]', 'Separator temperature [C]', 'X-averaged separator temperature [C]', 'Positive electrode temperature [C]', 'X-averaged p

In [7]:
model.default_parameter_values.search("temperature")

Results for 'temperature': ['Reference temperature [K]', 'Ambient temperature [K]', 'Initial temperature [K]']
Reference temperature [K] -> 298.15
Ambient temperature [K] -> 298.15
Initial temperature [K] -> 298.15


In [15]:
cc_stages = [(0.7, 4.2), (0.3, 4.2)]
exp = []

for c_rate, v_cut in cc_stages:
    exp.append(pybamm.step.c_rate(c_rate, termination=["Voltage [V]", f"{v_cut} V"]))
print(exp)

ValueError: could not convert string to float: ''

In [12]:
def evaluate_charge_protocol(cc_stages, start_soc=0.1, target_soc=0.8):
    """Return KPIs (+ soc/plating traces) of a multi-stage CC + CV charge on a thermal DFN."""
    option = {"thermal": "lumped"}
    model = pybamm.lithium_ion.DFN(options=option)
    parameter_values = pybamm.ParameterValues("Chen2020")
    exp = []
    for c_rate, v_cut in cc_stages:
        exp.append(pybamm.step.c_rate(c_rate, termination=["Voltage [V] >= {}".format(v_cut)]))
    exp.append("Hold at 4.2V until C/20")
    experiment = pybamm.Experiment(
        [
            tuple(exp),
        ])
    sim = pybamm.Simulation(model, parameter_values=parameter_values,experiment=experiment)
    sol = sim.solve(initial_soc=start_soc)
    bsoc = sol["Throughput capacity [A.h]"].data / parameter_values["Nominal cell capacity [A.h]"] + start_soc
    t = sol["Time [s]"].data / 60
    t_target = np.interp(target_soc, soc, t)
    # t_cc_end = sol.cycles[0].steps[0]["Time [s]"].data[-1] / 60
    # cv_share = (max(0,t_target - t_cc_end) / t_target)*100
    eta = sol['Negative electrode surface potential difference at separator interface [V]'].entries
    T_cell = sol["Cell temperature [K]"].data
    T_init = parameter_values["Initial temperature [K]"]
    temp_rise = T_cell.max() - T_init
    return {"time_to_target_min": t_target, "min_plating_mV": 1000*eta.min(), "temp_rise_K": temp_rise,
            "soc": bsoc, "plating_mV": eta*1000}

_Run the cell below, then compare your output to the target output already shown beneath it._

In [9]:
protocols = {
    "0.7C CC-CV (datasheet)": [(0.7, 4.2)],
    "1.0C CC-CV": [(1.0, 4.2)],
    "Ladder 1.5C/1.0C/0.5C": [(1.5, 3.9), (1.0, 4.05), (0.5, 4.2)],
}
results = {name: evaluate_charge_protocol(stages) for name, stages in protocols.items()}

summary = pd.DataFrame({
    name: {"t(10-80%) [min]": round(r["time_to_target_min"], 1),
           "plating margin [mV]": round(r["min_plating_mV"], 1),
           "temp rise [K]": round(r["temp_rise_K"], 2)}
    for name, r in results.items()
}).T
print(summary)

fig, ax = plt.subplots(figsize=(7, 4))
for name, r in results.items():
    ax.plot(r["soc"], r["plating_mV"], label=name)
ax.axhline(0, color="k", lw=0.8, ls="--")
ax.set_xlabel("SOC [-]")
ax.set_ylabel(r"$\phi_s - \phi_e$ at anode/separator [mV]")
ax.set_title("Plating margin during charge")
ax.legend()
plt.tight_layout()
plt.show()

AttributeError: 'float' object has no attribute 'get_event'

## 2. Cell design optimisation

Energy density pulls towards thick, dense electrodes; power and fast charging
pull the other way, because the electrolyte has to carry the current through
the whole electrode thickness. These problems use the DFN with `Chen2020` as the
baseline and scale the design around it. The rule used throughout: **whenever
you change the positive electrode's capacity, scale the negative electrode by
the same factor** so the N:P balance (and the eSOH windows) stay unchanged.

#### Problem 4: Electrode thickness vs. specific energy at different discharge rates

**Why it matters.** Thicker electrodes dilute the inactive mass (current collectors,
separator), so stack-level specific energy rises with thickness at low rate. At high
rate, electrolyte depletion across the thick electrode strands capacity, and the
optimum moves to thinner designs. Finding that optimum for the customer's duty cycle
is a day-one design question.

**Physics.** Scale both electrode thicknesses by the same factor $s$ (N:P stays
constant). Delivered energy $E = \int V I\,dt$. Stack areal mass per repeating unit
(double-side-coated current collectors, so each unit gets half of each foil), including
the electrolyte that fills the pores:

$$m'' = \sum_{k\in\{n,s,p\}} L_k\left(\rho_k + \varepsilon_k\,\rho_{el}\right)
+ \tfrac12 L_{cc,n}\rho_{cc,n} + \tfrac12 L_{cc,p}\rho_{cc,p},
\qquad e = \frac{E}{A\,m''}\ \ [\mathrm{Wh/kg}]$$

with $\rho_{el} = 1200\ \mathrm{kg\,m^{-3}}$ and $A$ = electrode height × width. The
`Chen2020` "density" values are dry-coating densities (solid only), which is why the
electrolyte term is added separately.

**Your task.** Write `stack_specific_energy(thickness_scale, c_rate)` returning the
stack specific energy in Wh/kg for a constant-current discharge at `c_rate` from 100 %
SOC to 2.5 V. Use the DFN and `Chen2020`, with both electrode thicknesses multiplied by
`thickness_scale`.

<details>
<summary><b>Hint 1</b>: the C-rate trap</summary>

The experiment converts "XC" to amps using `"Nominal cell capacity [A.h]"`. If you make
the electrodes 2x thicker but leave the nominal capacity alone, "1C" is really C/2 for
your new design. Scale it too.

</details>

<details>
<summary><b>Hint 2</b>: energy</summary>

`"Discharge energy [W.h]"` stays zero unless the model is created with the option
`"calculate discharge energy": "true"`. Integrating `"Terminal voltage [V]"` ×
`"Current [A]"` over `"Time [s]"` with `np.trapezoid` is simpler and model-independent.

</details>

<details>
<summary><b>Hint 3</b>: parameter names</summary>

Use the pattern `f"{domain} electrode thickness [m]"`, `"... density [kg.m-3]"`,
`"... porosity"` with `domain` in `("Negative", "Positive")`, plus the separator and the
two current collectors. A small loop keeps the mass calculation tidy.

</details>

In [ ]:
def stack_specific_energy(thickness_scale, c_rate):
    """Return stack-level specific energy [Wh/kg] of a thickness-scaled Chen2020 cell."""
    # TODO: implement (see the cell above)
    return 1.0

_Run the cell below, then compare your output to the target output already shown beneath it._

In [ ]:
scales = [0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0]
rates = {"C/3": 1 / 3, "1C": 1.0, "2C": 2.0}
energy = pd.DataFrame({label: [stack_specific_energy(s, c) for s in scales] for label, c in rates.items()},
                      index=pd.Index(scales, name="thickness scale"))
print("Stack specific energy [Wh/kg]:")
print(energy.round(1))
print()
for label in rates:
    best = energy[label].idxmax()
    print(f"{label:>4}: best thickness scale = {best:4.2f} "
          f"(positive electrode {75.6 * best:5.1f} um) -> {energy[label].max():6.1f} Wh/kg")

ax = energy.plot(marker="o", figsize=(7, 4))
ax.set_ylabel("Stack specific energy [Wh/kg]")
ax.set_title("Thickness vs. rate trade-off (Chen2020, DFN)")
plt.tight_layout()
plt.show()

#### Problem 5: Porosity × thickness DOE and the energy-power Pareto front

**Why it matters.** Calendering density (porosity) and coat weight (thickness) are the
two main process knobs on a cathode. Lower porosity packs more active material into the
same volume but chokes electrolyte transport. A small DOE over both, reduced to a
**Pareto front** of energy vs. rate capability, is what you bring to a design review.

**Physics.** Change the positive electrode porosity $\varepsilon_p$ and set its active
material fraction to $1-\varepsilon_p$ (in `Chen2020` they sum to 1, i.e. binder and
carbon are lumped into the solid). The positive capacity then scales by

$$k = s\,\frac{1-\varepsilon_p}{1-\varepsilon_{p,0}}$$

where $s$ is the positive thickness scale. To keep N:P constant, scale the negative
thickness and the nominal capacity by the same $k$. KPIs:

$$\text{energy KPI} = e_{C/3}\ [\mathrm{Wh/kg}]\ \text{(same mass model as Problem 4)},\qquad
\text{retention}_{2C} = \frac{Q_{2C}}{Q_{C/3}}\times 100\ \%$$

**Your task.** Write `design_kpis(pos_porosity, thickness_scale)` returning
`{"wh_per_kg": ..., "retention_2c_pct": ...}`. Use the DFN and `Chen2020`, and discharge
from 100 % SOC to 2.5 V at C/3 and at 2C. You can copy your areal-mass code from
Problem 4 into a helper.

The test cell runs a 3 × 4 DOE and flags the Pareto-optimal designs (no other design
is better on *both* KPIs). A pure Pareto filter keeps extreme corners, such as the
highest-energy design with almost no 2C capability. In a real review you would add a
minimum-retention constraint (e.g. ≥ 80 %) before choosing.

<details>
<summary><b>Hint 1</b>: which parameters change</summary>

Four entries change: positive porosity, positive active material volume fraction,
positive thickness (× `thickness_scale`), negative thickness (× $k$) and nominal
capacity (× $k$). Read the baseline active fraction *before* you overwrite it. The
electrolyte mass in the positive electrode now changes with porosity.

</details>

<details>
<summary><b>Hint 2</b>: two runs, one parameter set</summary>

Build the parameter set once, then run two separate simulations (C/3 and 2C). Use the
last value of `"Discharge capacity [A.h]"` for $Q$.

</details>

In [ ]:
def design_kpis(pos_porosity, thickness_scale):
    """Return {"wh_per_kg", "retention_2c_pct"} for a porosity/thickness-modified Chen2020 cell."""
    # TODO: implement (see the cell above)
    return {"wh_per_kg": 1.0, "retention_2c_pct": 1.0}

_Run the cell below, then compare your output to the target output already shown beneath it._

In [ ]:
rows = []
for eps in [0.25, 0.335, 0.45]:
    for s in [0.75, 1.0, 1.5, 2.0]:
        k = design_kpis(eps, s)
        rows.append({"porosity": eps, "thickness scale": s,
                     "Wh/kg @C/3": k["wh_per_kg"], "2C retention [%]": k["retention_2c_pct"]})
doe = pd.DataFrame(rows)

# Pareto filter (scaffolding): a design is dominated if another one is >= on both KPIs and > on one
e, r = doe["Wh/kg @C/3"].to_numpy(), doe["2C retention [%]"].to_numpy()
doe["Pareto"] = [not np.any((e >= e[i]) & (r >= r[i]) & ((e > e[i]) | (r > r[i]))) for i in range(len(doe))]
print(doe.round({"Wh/kg @C/3": 1, "2C retention [%]": 1}).to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 4))
for eps, grp in doe.groupby("porosity"):
    ax.plot(grp["Wh/kg @C/3"], grp["2C retention [%]"], "o-", label=f"porosity {eps}")
front = doe[doe["Pareto"]].sort_values("Wh/kg @C/3")
ax.plot(front["Wh/kg @C/3"], front["2C retention [%]"], "k--", lw=1, label="Pareto front")
ax.set_xlabel("Stack specific energy at C/3 [Wh/kg]")
ax.set_ylabel("2C / C/3 capacity retention [%]")
ax.legend()
plt.tight_layout()
plt.show()

#### Problem 6: Maximum plating-free areal loading for a fast-charge spec

**Why it matters.** The customer spec reads "10-80 % at X C, no plating", and they
want maximum energy. Thicker electrodes give more energy but lose plating margin. This
problem inverts the question: **for each charge C-rate, what is the thickest (highest
areal capacity) design that still charges plating-free?** The result, areal loading
vs. charge rate, is the fast-charge counterpart of a Ragone plot, and a standard input
to platform decisions.

**Physics.** Same co-scaled design as Problem 4 (both electrodes and the nominal
capacity × $s$), same plating criterion as Problem 2 (isothermal DFN, 25 °C). The
charge is the spec window: constant current from 10 % SOC for the time it takes to add
70 % SOC at that rate, **or until 4.2 V, whichever comes first**:

$$t_{spec} = \frac{0.7 \times 3600}{C}\ \text{[s]},\qquad
Q'' = \frac{Q_{nom}(s)}{A}\ \text{reported in mAh/cm}^2$$

**Your task.** Write `fast_charge_design_check(thickness_scale, c_rate, start_soc=0.1)`
returning a dict with:

* `"plating_margin_mV"`: min $\phi_s-\phi_e$ at the anode/separator interface [mV]
* `"areal_capacity_mAh_cm2"`: $Q''$ of the scaled design
* `"soc_end"`: SOC at the end of the step (below 0.8 means the cell hit 4.2 V first)

The test cell uses `brentq` on the thickness scale to find where the margin crosses
0 mV.

<details>
<summary><b>Hint 1</b>: reuse, and the step string</summary>

This is Problem 2's simulation on Problem 4's scaled parameter set. A single
experiment step can end on whichever condition comes first:
`"... for <duration> seconds or until <voltage> V"`.

</details>

<details>
<summary><b>Hint 2</b>: units and sanity check</summary>

Convert Ah → mAh and m² → cm². The baseline `Chen2020` electrode is about
4.9 mAh/cm². The plating-free loading should drop steeply as the charge rate goes up;
high-power cells sit around 1-2 mAh/cm² for exactly this reason.

</details>

<details>
<summary><b>Hint 3</b>: why the test stops at 1C</summary>

Past about 1.2C, even a very thin electrode plates (and hits 4.2 V before 80 % SOC).
Then the limit is **particle-level** (kinetics and solid diffusion), which does not
improve with thinner coatings, so there is no root to find. The last line of the test
cell shows this.

</details>

In [ ]:
def fast_charge_design_check(thickness_scale, c_rate, start_soc=0.1):
    """Return {"plating_margin_mV", "areal_capacity_mAh_cm2", "soc_end"} for a co-scaled Chen2020 design."""
    # TODO: implement (see the cell above)
    return {"plating_margin_mV": 1.0, "areal_capacity_mAh_cm2": 1.0, "soc_end": 0.8}

_Run the cell below, then compare your output to the target output already shown beneath it._

In [ ]:
rows = []
for c in [0.5, 0.75, 1.0]:
    try:
        s_max = brentq(lambda s: fast_charge_design_check(s, c)["plating_margin_mV"], 0.15, 2.0, xtol=1e-3)
        res = fast_charge_design_check(s_max, c)
        rows.append({"charge C-rate": c, "thickness scale": round(s_max, 3),
                     "positive thickness [um]": round(75.6 * s_max, 1),
                     "areal capacity [mAh/cm2]": round(res["areal_capacity_mAh_cm2"], 2),
                     "SOC reached": round(res["soc_end"], 3)})
    except ValueError as e:
        print(f"{c}C: could not bracket a root (function not implemented yet?): {e}")
limits = pd.DataFrame(rows)
print(limits.to_string(index=False))

thin = fast_charge_design_check(0.15, 1.5)
print(f"\nThinnest design (scale 0.15) at 1.5C: margin {thin['plating_margin_mV']:.1f} mV, "
      f"SOC reached {thin['soc_end']:.3f} -> particle-limited, no thickness fixes it")

if len(limits):
    ax = limits.plot(x="charge C-rate", y="areal capacity [mAh/cm2]", marker="o", legend=False, figsize=(6, 4))
    ax.set_ylabel("Max plating-free areal capacity [mAh/cm$^2$]")
    ax.set_title("Plating-limited loading, 10-80 % CC (isothermal DFN, 25 degC)")
    plt.tight_layout()
    plt.show()

## 3. Parameter estimation

Every model is only as good as its parameters. This section covers the standard
estimation workflows: fitting an equivalent-circuit model to a pulse,
extracting solid diffusivity from GITT (first with the classic analytical
formula, then by fitting the physics model directly), and decomposing capacity
fade into degradation modes from a slow-rate curve. Each problem starts from
**Virtual lab** data with a known truth, so you can see how well each method
recovers it.

#### Problem 7: Fitting a Thevenin ECM to a pulse measured on the 'real' cell

**Why it matters.** BMS models and system engineers live on equivalent-circuit models.
Fitting R0/R1/C1 to pulse data is the most common parameter-estimation task in
industry. Here you fit a **Thevenin (1-RC) ECM** in PyBaMM to a 2C, 30 s discharge pulse
"measured" on a DFN cell (run the Virtual lab cell first).

**Physics.**

$$V(t) = U_{ocv}(z) - I R_0 - V_1,\qquad \frac{dV_1}{dt} = \frac{I}{C_1} - \frac{V_1}{R_1 C_1},
\qquad \tau_1 = R_1 C_1$$

Over a 30 s pulse the SOC moves by about 1.7 %, so use a linear OCV around 50 % SOC:
$U_{ocv}(z) = \mathrm{OCV\_50} + \mathrm{OCV\_SLOPE}\,(z-0.5)$.

**Your task.** Write `thevenin_voltage(t, r0, r1, c1)`: simulate PyBaMM's Thevenin ECM
driven by the measured current `i_lab` (given at `t_lab`) and return the model
voltage at the times `t`. It must have this exact signature, because the test cell
passes it straight to `scipy.optimize.curve_fit`. Use the constants from the Virtual
lab cell: start at 50 % SOC, capacity `Q_CELL`, constant R0/R1/C1.

<details>
<summary><b>Hint 1</b>: model and parameters</summary>

The model lives in `pybamm.equivalent_circuit`, and its default parameter set is
`"ECM_Example"`. Print its keys: `"R0 [Ohm]"`, `"R1 [Ohm]"`, `"C1 [F]"` default to
functions of (T, I, SOC), but plain floats are accepted. `"Open-circuit voltage [V]"`
expects a function of one argument (SOC), and ordinary arithmetic works on symbolic
inputs. Set both `"Cell capacity [A.h]"` and `"Nominal cell capacity [A.h]"`, plus
`"Initial SoC"`.

</details>

<details>
<summary><b>Hint 2</b>: a measured current profile</summary>

A tabulated current goes in as
`"Current function [A]": pybamm.Interpolant(t_lab, i_lab, pybamm.t)`. Positive current
means discharge in both the DFN data and the ECM, so no sign flip is needed.

</details>

<details>
<summary><b>Hint 3</b>: two traps when returning the voltage</summary>

1. Solve with the **full** time array (`t_eval=t`), not `[t0, tf]`. Otherwise the
   solver only stores the end points and the RC state comes back as a straight line.
2. The returned solution has extra points where the current steps, so
   `sol["Voltage [V]"].entries` will not have the same length as `t`. Call the
   processed variable *at* your times instead: `sol["Voltage [V]"](t)`.

</details>

**Virtual lab**: given scaffolding that generates the "measured" data. Run it as is.

In [ ]:
# ---- Virtual lab (scaffolding): HPPC-style 2C pulse on a fresh LG M50 at 50 % SOC ----
# The "real cell" is a DFN with Chen2020; 0.5 mV Gaussian noise mimics the cycler.
lab_params = pybamm.ParameterValues("Chen2020")
lab_exp = pybamm.Experiment(
    [("Rest for 60 seconds", "Discharge at 2C for 30 seconds", "Rest for 300 seconds")],
    period="1 second",
)
lab_sol = pybamm.Simulation(pybamm.lithium_ion.DFN(), parameter_values=lab_params,
                            experiment=lab_exp).solve(initial_soc=0.5)
t_lab, idx = np.unique(lab_sol["Time [s]"].entries, return_index=True)  # drop duplicate step-boundary times
i_lab = lab_sol["Current [A]"].entries[idx]
v_lab = lab_sol["Terminal voltage [V]"].entries[idx] + np.random.default_rng(0).normal(0, 0.5e-3, idx.size)

OCV_50 = lab_sol["Terminal voltage [V]"].entries[0]  # rested voltage at 50 % SOC [V]
OCV_SLOPE = 0.93    # dOCV/dSOC around 50 % SOC [V per unit SOC], from the C/20 OCV test
Q_CELL = 5.0        # capacity [A.h]
print(f"{t_lab.size} samples, OCV(50%) = {OCV_50:.4f} V, min voltage = {v_lab.min():.4f} V")

In [ ]:
def thevenin_voltage(t, r0, r1, c1):
    """Return the 1-RC Thevenin ECM voltage [V] at times t for the measured lab current."""
    # TODO: implement (see the cell above)
    return np.full(np.shape(t), 3.7) - r0 - r1 - 1e-6 * c1

_Run the cell below, then compare your output to the target output already shown beneath it._

In [ ]:
p0 = [0.02, 0.01, 1000.0]
bounds = ([1e-4, 1e-4, 10.0], [0.1, 0.1, 1e5])
popt, pcov = curve_fit(thevenin_voltage, t_lab, v_lab, p0=p0, bounds=bounds, x_scale=[0.01, 0.01, 1000.0])
r0, r1, c1 = popt
sigma = np.sqrt(np.diag(pcov))
v_fit = thevenin_voltage(t_lab, *popt)
rmse_mv = 1000 * np.sqrt(np.mean((v_fit - v_lab) ** 2))
print(f"R0  = {1000 * r0:6.2f} +/- {1000 * sigma[0]:.2f} mOhm")
print(f"R1  = {1000 * r1:6.2f} +/- {1000 * sigma[1]:.2f} mOhm")
print(f"C1  = {c1:8.1f} +/- {sigma[2]:.1f} F")
print(f"tau = {r1 * c1:6.1f} s")
print(f"RMSE = {rmse_mv:.2f} mV")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 5), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
ax1.plot(t_lab, v_lab, ".", ms=2, label="lab (DFN + noise)")
ax1.plot(t_lab, v_fit, "-", label="Thevenin fit")
ax1.set_ylabel("Voltage [V]")
ax1.legend()
ax2.plot(t_lab, 1000 * (v_fit - v_lab), "-", lw=0.8)
ax2.set_ylabel("Residual [mV]")
ax2.set_xlabel("Time [s]")
plt.tight_layout()
plt.show()

#### Problem 8: GITT and Weppner-Huggins: validate the method on a virtual cell

**Why it matters.** GITT is the standard way to measure solid-state diffusivity
$D(\text{sto})$. Most labs then apply the **Weppner-Huggins (WH)** formula. Before you
trust WH numbers from a real half-cell, check on a virtual cell, where you know the
true $D$, how biased the formula is for *your* pulse length.

**Physics.** For a spherical particle of radius $R_p$ (volume/surface $= R_p/3$) and a
pulse of duration $\tau$:

$$D_{WH} = \frac{4}{\pi\tau}\left(\frac{R_p}{3}\right)^2\left(\frac{\Delta E_s}{\Delta E_t}\right)^2,
\qquad \text{valid for } \tau \ll R_p^2/D$$

* $\Delta E_s$: change of the *rested* voltage across one pulse (end of this rest
  minus end of the previous rest).
* $\Delta E_t$: voltage change *during* the pulse, excluding the IR step (last pulse
  sample minus first pulse sample).

**Your task.** Write `gitt_weppner_huggins(pulse_s=600, n_pulses=25)` that:

1. simulates GITT on an NMC532 **half-cell** (`Xu2019`, SPM with option
   `{"working electrode": "positive"}`): an initial 10 min rest, then `n_pulses` ×
   (discharge at 0.2C for `pulse_s` seconds, rest 1 hour), sampled every 10 s;
2. applies WH to every pulse.

Return `{"sto": array, "D_wh": array, "D_true": float}`. Here `sto` is the average
positive particle stoichiometry at the end of each rest, and `D_true` is the
diffusivity in the parameter set.

<details>
<summary><b>Hint 1</b>: experiment structure</summary>

Make the initial rest its own one-step cycle and each pulse + rest a two-step cycle:
`[("Rest ...",)] + [(pulse, rest)] * n_pulses`. Then `sol.cycles[0]` is the initial rest
and `sol.cycles[k].steps[0]` / `.steps[1]` are pulse *k* and its rest. Use
`period="10 seconds"` in the `Experiment`.

</details>

<details>
<summary><b>Hint 2</b>: half-cell variables</summary>

The half-cell model reports `"Voltage [V]"`. Search `model.variables` for
`"stoichiometry"` to find the average positive particle stoichiometry.
`R_p` and `D_true` come from `"Positive particle radius [m]"` and
`"Positive particle diffusivity [m2.s-1]"`.

</details>

In [ ]:
def gitt_weppner_huggins(pulse_s=600, n_pulses=25):
    """Return {"sto", "D_wh", "D_true"} from a simulated GITT on the Xu2019 half-cell."""
    # TODO: implement (see the cell above)
    return {"sto": np.linspace(0.1, 0.5, 3), "D_wh": np.ones(3), "D_true": 1.0}

_Run the cell below, then compare your output to the target output already shown beneath it._

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for pulse_s in [600, 150]:
    r = gitt_weppner_huggins(pulse_s=pulse_s)
    ratio = r["D_wh"] / r["D_true"]
    print(f"pulse {pulse_s:3d} s: sto {r['sto'][0]:.3f}-{r['sto'][-1]:.3f}, "
          f"median D_WH / D_true = {np.median(ratio):.3f}, first-pulse ratio = {ratio[0]:.3f}")
    ax.semilogy(r["sto"], r["D_wh"], "o-", ms=3, label=f"Weppner-Huggins, {pulse_s} s pulses")
ax.axhline(r["D_true"], color="k", ls="--", label="true D (parameter set)")
ax.set_xlabel("Positive particle stoichiometry [-]")
ax.set_ylabel(r"$D$ [m$^2$/s]")
ax.legend()
plt.tight_layout()
plt.show()

#### Problem 9: Model-based GITT fit: diffusivity and kinetics together

**Why it matters.** Problem 8 showed that WH is biased. The modern alternative is to
fit the physics model directly to the GITT voltage trace. It uses every sample (not
two numbers per pulse), has no $\tau \ll R^2/D$ restriction, and can identify the
kinetics at the same time. Kinetics show up mostly in the instantaneous IR step and
diffusion in the $\sqrt t$ transient, so the two are *partly* separable. The covariance
from the fit tells you how well.

**Physics.** Two unknowns, fitted in log space because they span decades:

$$\theta = \left[\log_{10} D_p,\ \log_{10} k\right],\qquad
j_{0,p}(c_e, c_{s,surf}, c_{s,max}, T) = k \cdot j_{0,p}^{\,Xu2019}(c_e, c_{s,surf}, c_{s,max}, T)$$

$$\hat\theta = \arg\min_\theta \sum_i \left(V_{model}(t_i;\theta) - V_{lab,i}\right)^2$$

**Your task.** Run the Virtual lab cell first. Then write
`gitt_model_voltage(t, log10_d, log10_k)`: simulate the **same** GITT protocol as the
lab (8 × [0.2C for 600 s, rest 30 min], `period="10 seconds"`, `Xu2019`, SPM half-cell)
with $D_p = 10^{\log_{10} D}$ and the exchange-current density scaled by
$10^{\log_{10} k}$. Return the voltage at times `t`. The test cell calls `curve_fit`.

<details>
<summary><b>Hint 1</b>: scaling a parameter *function*</summary>

The exchange-current density is a Python function of `(c_e, c_s_surf, c_s_max, T)`.
Read the original function from the parameter set into a variable first, then assign a
new `lambda` with the same four arguments that multiplies it. If you skip the first
step, the lambda ends up calling itself.

</details>

<details>
<summary><b>Hint 2</b>: returning the voltage at the lab times</summary>

The same rule as Problem 7 applies: evaluate the processed variable at the requested
times with `sol["Voltage [V]"](t)`.

</details>

**Virtual lab**: given scaffolding that generates the "measured" data. Run it as is.

In [ ]:
# ---- Virtual lab (scaffolding): GITT on an NMC532 half-cell with "unknown" properties ----
TRUE_D = 2.5e-14     # positive particle diffusivity used to make the data [m2/s]
TRUE_K = 0.3         # multiplier on the Xu2019 positive exchange-current density
lab_gitt_params = pybamm.ParameterValues("Xu2019")
lab_gitt_params["Positive particle diffusivity [m2.s-1]"] = TRUE_D
_j0_ref = lab_gitt_params["Positive electrode exchange-current density [A.m-2]"]
lab_gitt_params["Positive electrode exchange-current density [A.m-2]"] = (
    lambda c_e, c_s_surf, c_s_max, T: TRUE_K * _j0_ref(c_e, c_s_surf, c_s_max, T))
gitt_exp = pybamm.Experiment([("Discharge at 0.2C for 600 seconds", "Rest for 30 minutes")] * 8,
                             period="10 seconds")
gitt_sol = pybamm.Simulation(pybamm.lithium_ion.SPM({"working electrode": "positive"}),
                             parameter_values=lab_gitt_params, experiment=gitt_exp).solve()
t_gitt, idx = np.unique(gitt_sol["Time [s]"].entries, return_index=True)
v_gitt = gitt_sol["Voltage [V]"].entries[idx] + np.random.default_rng(1).normal(0, 1e-3, idx.size)
print(f"{t_gitt.size} samples over {t_gitt[-1] / 3600:.1f} h, voltage {v_gitt.min():.3f}-{v_gitt.max():.3f} V")

In [ ]:
def gitt_model_voltage(t, log10_d, log10_k):
    """Return the SPM half-cell GITT voltage [V] at times t for given log10(D_p) and log10(j0 scale)."""
    # TODO: implement (see the cell above)
    return np.full(np.shape(t), 4.0) + 1e-3 * (log10_d + log10_k)

_Run the cell below, then compare your output to the target output already shown beneath it._

In [ ]:
popt, pcov = curve_fit(gitt_model_voltage, t_gitt, v_gitt, p0=[-14.0, 0.0],
                       bounds=([-16.0, -2.0], [-12.0, 2.0]), diff_step=1e-3)
sig = np.sqrt(np.diag(pcov))
with np.errstate(invalid="ignore", divide="ignore"):
    corr = pcov[0, 1] / (sig[0] * sig[1])
d_fit, k_fit = 10 ** popt
v_fit = gitt_model_voltage(t_gitt, *popt)
print(f"D_p fit  = {d_fit:.3e} m2/s (true {TRUE_D:.1e}), 1-sigma in log10 = {sig[0]:.4f}")
print(f"j0 scale = {k_fit:.3f}       (true {TRUE_K}),     1-sigma in log10 = {sig[1]:.4f}")
print(f"correlation(log10 D, log10 k) = {corr:.2f}")
print(f"RMSE = {1000 * np.sqrt(np.mean((v_fit - v_gitt) ** 2)):.2f} mV")

mask = t_gitt < 2 * (600 + 1800)  # first two pulses
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(t_gitt[mask] / 60, v_gitt[mask], ".", ms=3, label="lab")
ax.plot(t_gitt[mask] / 60, v_fit[mask], "-", label="model fit")
ax.set_xlabel("Time [min]")
ax.set_ylabel("Voltage [V]")
ax.legend()
plt.tight_layout()
plt.show()

#### Problem 10: Degradation-mode analysis (LLI, LAM_NE, LAM_PE) from a C/20 discharge

**Why it matters.** "The cell lost 7 % capacity" is not actionable. *Why* it lost it is:
lithium inventory (SEI growth, plating), or anode or cathode active material (cracking,
dissolution). Degradation-mode analysis (DMA) fits the aged cell's slow-rate voltage
curve with the two **fresh electrode OCPs**, stretched and shifted. It is the
standard non-destructive diagnosis on cells returned from the field.

**Physics.** During discharge from full ($Q$ = Ah discharged):

$$x(Q) = x_{100} - \frac{Q}{C_n},\qquad y(Q) = y_{100} + \frac{Q}{C_p},\qquad
V_{ocv}(Q) = U_p\big(y(Q)\big) - U_n\big(x(Q)\big)$$

$C_n, C_p$ [Ah] are the electrode capacities per unit stoichiometry:
$C = \varepsilon_s L A\, c_{max} F / 3600$. The lithium inventory in Ah is
$n_{Li} = x_{100} C_n + y_{100} C_p$. Comparing fitted (aged) and fresh values:

$$\mathrm{LAM_{NE}} = 1-\frac{C_n}{C_n^{fresh}},\quad
\mathrm{LAM_{PE}} = 1-\frac{C_p}{C_p^{fresh}},\quad
\mathrm{LLI} = 1-\frac{n_{Li}}{n_{Li}^{fresh}}$$

**Your task (two functions).** Run the Virtual lab cell first.

1. `fresh_electrode_inventory()`: for the **fresh** `Chen2020` cell, return
   `{"cap_neg_ah", "cap_pos_ah", "x100", "y100"}` ($C_n$, $C_p$ and the 100 % SOC
   stoichiometries from PyBaMM's electrode-SOH solver).
2. `full_cell_ocv(q_ah, cap_neg_ah, cap_pos_ah, x100, y100)`: return $V_{ocv}$ at the
   discharged capacities `q_ah` (NumPy array), using the `Chen2020` electrode OCP
   functions. The test cell fits it to the lab curve with `curve_fit`.

<details>
<summary><b>Hint 1</b>: fresh inventory</summary>

`pybamm.lithium_ion.electrode_soh.get_min_max_stoichiometries(params, param=model.param)`
returns `x_0, x_100, y_100, y_0` (any lithium-ion `model`, e.g. SPM). For $C_n$/$C_p$
use the active material volume fraction, thickness, `"Electrode height [m]"` ×
`"Electrode width [m]"`, the maximum concentration and `pybamm.constants.F.value`.

</details>

<details>
<summary><b>Hint 2</b>: evaluating the OCPs</summary>

`params["Negative electrode OCP [V]"]` (and the positive one) return plain Python
functions of stoichiometry. The `Chen2020` ones accept NumPy arrays directly. Clip the
stoichiometries into (0, 1) first: the optimiser will try out-of-range values.

</details>

<details>
<summary><b>Hint 3</b>: expect a small bias</summary>

The lab curve is C/20, not true OCV, so its overpotential is a few mV. You should
recover the modes to within about 1 percentage point, not exactly. In practice you
would average a C/20 charge and discharge to cancel this.

</details>

**Virtual lab**: given scaffolding that generates the "measured" data. Run it as is.

In [ ]:
# ---- Virtual lab (scaffolding): C/20 discharge of an AGED LG M50 ----
# Ageing applied: 10 % loss of negative active material, 5 % positive, 8 % lithium inventory.
LAM_NE_TRUE, LAM_PE_TRUE, LLI_TRUE = 0.10, 0.05, 0.08
aged = pybamm.ParameterValues("Chen2020")
aged["Negative electrode active material volume fraction"] *= 1 - LAM_NE_TRUE
aged["Positive electrode active material volume fraction"] *= 1 - LAM_PE_TRUE
# electrode stoichiometries of the aged cell when charged to 4.2 V (pre-computed)
aged["Initial concentration in negative electrode [mol.m-3]"] = 0.917675 * aged["Maximum concentration in negative electrode [mol.m-3]"]
aged["Initial concentration in positive electrode [mol.m-3]"] = 0.263845 * aged["Maximum concentration in positive electrode [mol.m-3]"]
aged_sol = pybamm.Simulation(pybamm.lithium_ion.SPM(), parameter_values=aged,
                             experiment=pybamm.Experiment(["Discharge at C/20 until 2.5 V"],
                                                          period="2 minutes")).solve()
q_aged = aged_sol["Discharge capacity [A.h]"].entries
v_aged = aged_sol["Terminal voltage [V]"].entries
print(f"Aged cell C/20 capacity: {q_aged[-1]:.3f} Ah ({q_aged.size} samples)")

In [ ]:
def fresh_electrode_inventory():
    """Return {"cap_neg_ah", "cap_pos_ah", "x100", "y100"} for the fresh Chen2020 cell."""
    # TODO: implement (see the cell above)
    return {"cap_neg_ah": 1.0, "cap_pos_ah": 1.0, "x100": 0.5, "y100": 0.5}


def full_cell_ocv(q_ah, cap_neg_ah, cap_pos_ah, x100, y100):
    """Return the full-cell pseudo-OCV [V] after discharging q_ah from the 100 % SOC state."""
    # TODO: implement (see the cell above)
    return np.full(np.shape(q_ah), 3.7) + 1e-3 * (cap_neg_ah + cap_pos_ah + x100 + y100)

_Run the cell below, then compare your output to the target output already shown beneath it._

In [ ]:
fresh = fresh_electrode_inventory()
print("Fresh: " + ", ".join(f"{k} = {v:.4f}" for k, v in fresh.items()))

p0 = [fresh["cap_neg_ah"], fresh["cap_pos_ah"], fresh["x100"], fresh["y100"]]
bounds = ([0.5, 0.5, 0.3, 0.05], [15.0, 15.0, 1.0, 0.9])
popt, _ = curve_fit(full_cell_ocv, q_aged, v_aged, p0=p0, bounds=bounds)
cn, cp, x100, y100 = popt
li_fresh = fresh["x100"] * fresh["cap_neg_ah"] + fresh["y100"] * fresh["cap_pos_ah"]
modes = {"LAM_NE": (1 - cn / fresh["cap_neg_ah"], LAM_NE_TRUE),
         "LAM_PE": (1 - cp / fresh["cap_pos_ah"], LAM_PE_TRUE),
         "LLI": (1 - (x100 * cn + y100 * cp) / li_fresh, LLI_TRUE)}
for name, (fit, true) in modes.items():
    print(f"{name:7s} fitted {100 * fit:5.2f} %   true {100 * true:5.2f} %")
v_fit = full_cell_ocv(q_aged, *popt)
print(f"RMSE = {1000 * np.sqrt(np.mean((v_fit - v_aged) ** 2)):.2f} mV")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.plot(q_aged, v_aged, ".", ms=3, label="aged cell, C/20")
ax1.plot(q_aged, v_fit, "-", label="DMA fit")
ax1.set_xlabel("Discharged capacity [Ah]")
ax1.set_ylabel("Voltage [V]")
ax1.legend()
# differential voltage (dV/dQ) is where the electrode features show up
ax2.plot(q_aged[1:-1], -np.gradient(v_aged, q_aged)[1:-1], ".", ms=3, label="aged cell")
ax2.plot(q_aged[1:-1], -np.gradient(v_fit, q_aged)[1:-1], "-", label="DMA fit")
ax2.set_ylim(0, 1.0)
ax2.set_xlabel("Discharged capacity [Ah]")
ax2.set_ylabel("-dV/dQ [V/Ah]")
ax2.legend()
plt.tight_layout()
plt.show()

## Wrap-up

**Fast-charge protocols**
* Problem 1: time-to-80 % saturates with C-rate because the CV phase takes over;
  quote both the time and the CV share.
* Problem 2: the anode/separator-interface potential $\phi_s-\phi_e$ is the standard
  plating criterion; `brentq` turns it into a plating-limited C-rate.
* Problem 3: a multi-stage CC ladder can beat a single CC-CV on both charge time and
  plating margin. Self-heating (lumped thermal) also helps the margin.

**Cell design optimisation**
* Problem 4: specific energy vs. thickness has a rate-dependent optimum, driven
  by electrolyte transport limits; keep N:P and the nominal capacity consistent
  when you scale.
* Problem 5: a porosity × thickness DOE plus a Pareto filter is the standard way
  to present an energy-power trade-off.
* Problem 6: the plating-limited areal loading vs. charge C-rate is the
  fast-charge counterpart of the Ragone plot.

**Parameter estimation**
* Problem 7: a 1-RC Thevenin fit to a DFN pulse gives R0/R1/C1. The residual
  shows what a single RC cannot capture (the slow diffusion tail).
* Problem 8: Weppner-Huggins is quick but biased when the pulse is not short
  compared with $R^2/D$. Validate it on a virtual cell before trusting it.
* Problem 9: a model-based fit recovers D and kinetics together; the parameter
  correlation tells you how independently they are identified.
* Problem 10: degradation-mode analysis (LLI, LAM_NE, LAM_PE) only needs the
  electrode OCPs and a slow-rate curve.

**Further reading**
* PyBaMM documentation: https://docs.pybamm.org/en/stable/
* SciPy `curve_fit`: https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.curve_fit.html
* SciPy `brentq`: https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.brentq.html